In [29]:
from google.colab import userdata
OPEN_API_KEY = userdata.get('OPENAI_API_KEY')

import os
os.environ['OPENAI_API_KEYS'] = OPEN_API_KEY


In [30]:
!pip install -qU "langchain-chroma>=0.1.2"

In [31]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [32]:
from langchain_core.documents import Document


# 1. Rohit Sharma
doc1 = Document(
    page_content="Rohit Sharma is one of the most successful captains and dynamic opening batsmen in IPL history, known for his effortless pull shots and tactical brilliance. He has led the MI franchise to multiple championship titles.",
    metadata={"team": "Mumbai Indians"}
)

# 2. MS Dhoni
doc2 = Document(
    page_content="MS Dhoni is one of the most iconic finishing batsmen and legendary captains in IPL history, known for his calm composure under pressure and lightning-fast wicketkeeping. He has led the CSK franchise across multiple successful eras.",
    metadata={"team": "Chennai Super Kings"}
)

# 3. Gautam Gambhir
doc3 = Document(
    page_content="Gautam Gambhir is one of the most gritty and clutch top-order batsmen in IPL history, known for his fiery leadership and excellent play against spin. He has led the KKR franchise to multiple IPL trophies.",
    metadata={"team": "Kolkata Knight Riders"}
)

# 4. David Warner
doc4 = Document(
    page_content="David Warner is one of the most destructive and consistent overseas opening batsmen in IPL history, known for his explosive powerplay scoring and intense energy on the field. He has led the SRH franchise to an IPL title.",
    metadata={"team": "Sunrisers Hyderabad"}
)

# 5. KL Rahul
doc5 = Document(
    page_content="KL Rahul is one of the most technically sound and versatile top-order batsmen in IPL history, known for his elegant strokeplay and reliable anchor role. He has led the LSG franchise in multiple consecutive playoff campaigns.",
    metadata={"team": "Lucknow Super Giants"}
)



In [33]:
docs = [doc1,doc2,doc3,doc4,doc5]

In [37]:
vector_store = Chroma(
    embedding_function = OpenAIEmbeddings(api_key=OPEN_API_KEY),
    persist_directory = 'my_chroma_db',
    collection_name = 'ipl_sample'
)

In [38]:
vector_store.add_documents(docs)

['fb00e2d6-c037-4ed6-a686-8f380648edd0',
 '568f60e5-2808-4c81-939d-54564285f533',
 '6365c549-5d48-4fe0-8e26-e54f61835c3c',
 '0a700fdd-1f91-4583-b0ff-ad977103dccc',
 '6f6efec2-6df5-4ef4-bea2-cca2efb62a5c']

In [40]:
vector_store.get(include=[
    'embeddings',
    'documents',
    'metadatas'
    ])

{'ids': ['fb00e2d6-c037-4ed6-a686-8f380648edd0',
  '568f60e5-2808-4c81-939d-54564285f533',
  '6365c549-5d48-4fe0-8e26-e54f61835c3c',
  '0a700fdd-1f91-4583-b0ff-ad977103dccc',
  '6f6efec2-6df5-4ef4-bea2-cca2efb62a5c'],
 'embeddings': array([[-1.19990846e-02, -3.10502644e-03,  9.72812623e-03, ...,
         -1.26497531e-02,  1.08572263e-02, -7.76974251e-03],
        [-9.73203965e-03,  7.68863456e-03,  1.65541668e-03, ...,
         -1.04950834e-02,  3.89604899e-03,  5.97825274e-03],
        [-1.40423486e-02, -6.94596529e-05,  7.41897477e-03, ...,
         -4.06817049e-02,  8.75823572e-03,  1.97905628e-03],
        [-7.65581755e-03, -1.39652668e-02,  3.67281237e-03, ...,
         -1.19985137e-02,  2.82803248e-03,  2.30334513e-02],
        [ 9.41155944e-03,  7.92340748e-03, -2.08810600e-03, ...,
         -2.68537812e-02,  5.60403429e-03,  1.33598568e-02]]),
 'documents': ['Rohit Sharma is one of the most successful captains and dynamic opening batsmen in IPL history, known for his effortless

In [42]:
data = vector_store.get(include=['embeddings', 'documents', 'metadatas'])

for doc_id, text, embedding in zip(data['ids'], data['documents'], data['embeddings']):
  print(f"Document ID : {doc_id}")
  print(f"Text Preview: {text[:100]}...")
  print(f"Embedding vector dimension: {len(embedding)}")
  print("."*40)

Document ID : fb00e2d6-c037-4ed6-a686-8f380648edd0
Text Preview: Rohit Sharma is one of the most successful captains and dynamic opening batsmen in IPL history, know...
Embedding vector dimension: 1536
........................................
Document ID : 568f60e5-2808-4c81-939d-54564285f533
Text Preview: MS Dhoni is one of the most iconic finishing batsmen and legendary captains in IPL history, known fo...
Embedding vector dimension: 1536
........................................
Document ID : 6365c549-5d48-4fe0-8e26-e54f61835c3c
Text Preview: Gautam Gambhir is one of the most gritty and clutch top-order batsmen in IPL history, known for his ...
Embedding vector dimension: 1536
........................................
Document ID : 0a700fdd-1f91-4583-b0ff-ad977103dccc
Text Preview: David Warner is one of the most destructive and consistent overseas opening batsmen in IPL history, ...
Embedding vector dimension: 1536
........................................
Document ID : 6f6efec2-6df5-

In [43]:
vector_store.similarity_search(
    query = 'who among them are bowlers',
    k= 2
)

[Document(id='0a700fdd-1f91-4583-b0ff-ad977103dccc', metadata={'team': 'Sunrisers Hyderabad'}, page_content='David Warner is one of the most destructive and consistent overseas opening batsmen in IPL history, known for his explosive powerplay scoring and intense energy on the field. He has led the SRH franchise to an IPL title.'),
 Document(id='6365c549-5d48-4fe0-8e26-e54f61835c3c', metadata={'team': 'Kolkata Knight Riders'}, page_content='Gautam Gambhir is one of the most gritty and clutch top-order batsmen in IPL history, known for his fiery leadership and excellent play against spin. He has led the KKR franchise to multiple IPL trophies.')]

In [45]:
vector_store.similarity_search_with_score(
    query = 'who won the ipl',
    k= 2
)

[(Document(id='fb00e2d6-c037-4ed6-a686-8f380648edd0', metadata={'team': 'Mumbai Indians'}, page_content='Rohit Sharma is one of the most successful captains and dynamic opening batsmen in IPL history, known for his effortless pull shots and tactical brilliance. He has led the MI franchise to multiple championship titles.'),
  0.4485558867454529),
 (Document(id='568f60e5-2808-4c81-939d-54564285f533', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni is one of the most iconic finishing batsmen and legendary captains in IPL history, known for his calm composure under pressure and lightning-fast wicketkeeping. He has led the CSK franchise across multiple successful eras.'),
  0.4539910554885864)]

In [46]:
#metadata filtering
vector_store.similarity_search_with_score(
    query = 'who won the ipl',
    k= 2,
    filter = {'team': 'Mumbai Indians'}
)


[(Document(id='fb00e2d6-c037-4ed6-a686-8f380648edd0', metadata={'team': 'Mumbai Indians'}, page_content='Rohit Sharma is one of the most successful captains and dynamic opening batsmen in IPL history, known for his effortless pull shots and tactical brilliance. He has led the MI franchise to multiple championship titles.'),
  0.4485558867454529)]

In [49]:
updated_doc1 = Document(
    page_content="Rohit Sharma is one of the most successful captains and dynamic opening batsmen in IPL history, known for his effortless pull shots and tactical brilliance. He has led the MI franchise to multiple championship titles.",
     metadata={"team": "Lucknow Super Giants"}
)

vector_store.update_document(
    document_id='fb00e2d6-c037-4ed6-a686-8f380648edd0',
    document=updated_doc1
)

In [50]:
vector_store.get(include=[
    'embeddings',
    'documents',
    'metadatas'
    ])

{'ids': ['fb00e2d6-c037-4ed6-a686-8f380648edd0',
  '568f60e5-2808-4c81-939d-54564285f533',
  '6365c549-5d48-4fe0-8e26-e54f61835c3c',
  '0a700fdd-1f91-4583-b0ff-ad977103dccc',
  '6f6efec2-6df5-4ef4-bea2-cca2efb62a5c'],
 'embeddings': array([[-1.19990846e-02, -3.10502644e-03,  9.72812623e-03, ...,
         -1.26497531e-02,  1.08572263e-02, -7.76974251e-03],
        [-9.73203965e-03,  7.68863456e-03,  1.65541668e-03, ...,
         -1.04950834e-02,  3.89604899e-03,  5.97825274e-03],
        [-1.40423486e-02, -6.94596529e-05,  7.41897477e-03, ...,
         -4.06817049e-02,  8.75823572e-03,  1.97905628e-03],
        [-7.65581755e-03, -1.39652668e-02,  3.67281237e-03, ...,
         -1.19985137e-02,  2.82803248e-03,  2.30334513e-02],
        [ 9.41155944e-03,  7.92340748e-03, -2.08810600e-03, ...,
         -2.68537812e-02,  5.60403429e-03,  1.33598568e-02]]),
 'documents': ['Rohit Sharma is one of the most successful captains and dynamic opening batsmen in IPL history, known for his effortless

In [51]:
vector_store.delete(ids = ['fb00e2d6-c037-4ed6-a686-8f380648edd0','568f60e5-2808-4c81-939d-54564285f533'])

In [52]:
vector_store.get(include=[
    'embeddings',
    'documents',
    'metadatas'
    ])

{'ids': ['6365c549-5d48-4fe0-8e26-e54f61835c3c',
  '0a700fdd-1f91-4583-b0ff-ad977103dccc',
  '6f6efec2-6df5-4ef4-bea2-cca2efb62a5c'],
 'embeddings': array([[-1.40423486e-02, -6.94596529e-05,  7.41897477e-03, ...,
         -4.06817049e-02,  8.75823572e-03,  1.97905628e-03],
        [-7.65581755e-03, -1.39652668e-02,  3.67281237e-03, ...,
         -1.19985137e-02,  2.82803248e-03,  2.30334513e-02],
        [ 9.41155944e-03,  7.92340748e-03, -2.08810600e-03, ...,
         -2.68537812e-02,  5.60403429e-03,  1.33598568e-02]]),
 'documents': ['Gautam Gambhir is one of the most gritty and clutch top-order batsmen in IPL history, known for his fiery leadership and excellent play against spin. He has led the KKR franchise to multiple IPL trophies.',
  'David Warner is one of the most destructive and consistent overseas opening batsmen in IPL history, known for his explosive powerplay scoring and intense energy on the field. He has led the SRH franchise to an IPL title.',
  'KL Rahul is one of t